

This notebook performs **data cleaning and merging** for bioassay data from **Tox21** and **NCATS** databases. The focus is on:

- **Mitochondrial membrane potential disruption** (cellular energy/toxicity)
- **Cell viability counter-screens** (general cytotoxicity)
- **Caspase-3/Caspase-3/7 induction** (apoptosis markers)

### Goal
Prepare a **dataset** for predicting toxicity of new molecules.

### Data Cleaning
1. Import & Inspection
2. Remove Unwanted Observations
3. Fix Structural Errors
4. Identify Column Data Types
5. Handle Missing Data
6. Manage Outliers
7. Assay Annotation (Domain-Aware)
8. Merging 
9. Data Validation & Verification
10. Final Output

---
# incluide 14.	Caspase-3/7 Activation Profiling of Human Lymphoblast Cell Lines

## Step 1: Import & Inspection

Import required libraries and load all datasets dynamically.

In [1]:
#import libraries

import pandas as pd
import numpy as np
import os
import warnings
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("Libraries imported successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully.
Pandas version: 2.2.2
NumPy version: 1.26.4


In [2]:
# base data directory
BASE_DIR = Path('Data')

# This maps AID numbers to metadata - descrption of assay 
ASSAY_METADATA = {
    # Mitochondrial Assays (Tox21)
    '720635': {
        'assay_name': 'AID_720635_mito_membrane_potential',
        'assay_type': 'mitochondrial',
        'cell_line': 'HepG2',
        'source': 'Tox21',
        'is_counter_screen': False,
        'biological_process': 'mitochondria',
        'endpoint_level': 'mechanistic'
    },
    '720634': {
        'assay_name': 'AID_720634_cell_viability',
        'assay_type': 'viability',
        'cell_line': 'HepG2',
        'source': 'Tox21',
        'is_counter_screen': True,
        'biological_process': 'cytotoxicity',
        'endpoint_level': 'general_toxicity'
    },
    # Tox21 Caspase-3/7 Assays
    '1346978': {
        'assay_name': 'AID_1346978_caspase37_HepG2',
        'assay_type': 'caspase',
        'cell_line': 'HepG2',
        'source': 'Tox21',
        'is_counter_screen': False,
        'biological_process': 'apoptosis',
        'endpoint_level': 'mechanistic'
    },
    '1346980': {
        'assay_name': 'AID_1346980_caspase37_CHO-K1',
        'assay_type': 'caspase',
        'cell_line': 'CHO-K1',
        'source': 'Tox21',
        'is_counter_screen': False,
        'biological_process': 'apoptosis',
        'endpoint_level': 'mechanistic'
    },
    '1346979': {
        'assay_name': 'AID_1346979_viability_CHO-K1',
        'assay_type': 'viability',
        'cell_line': 'CHO-K1',
        'source': 'Tox21',
        'is_counter_screen': True,
        'biological_process': 'cytotoxicity',
        'endpoint_level': 'general_toxicity'
    },
    '1346981': {
        'assay_name': 'AID_1346981_viability_HepG2',
        'assay_type': 'viability',
        'cell_line': 'HepG2',
        'source': 'Tox21',
        'is_counter_screen': True,
        'biological_process': 'cytotoxicity',
        'endpoint_level': 'general_toxicity'
    },
    # NCATS Caspase-3 Assays (13 cell lines4
    
    '654': {'assay_name': 'AID_654_caspase3_SHSY5Y', 'assay_type': 'caspase', 'cell_line': 'SH-SY5Y', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '655': {'assay_name': 'AID_655_caspase3_SK-N-SH', 'assay_type': 'caspase', 'cell_line': 'SK-N-SH', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '656': {'assay_name': 'AID_656_caspase3_N2a', 'assay_type': 'caspase', 'cell_line': 'N2a', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '657': {'assay_name': 'AID_657_caspase3_HepG2', 'assay_type': 'caspase', 'cell_line': 'HepG2', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '658': {'assay_name': 'AID_658_caspase3_BJ', 'assay_type': 'caspase', 'cell_line': 'BJ', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '659': {'assay_name': 'AID_659_caspase3_NIH-3T3', 'assay_type': 'caspase', 'cell_line': 'NIH-3T3', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '660': {'assay_name': 'AID_660_caspase3_HUV-EC-C', 'assay_type': 'caspase', 'cell_line': 'HUV-EC-C', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '661': {'assay_name': 'AID_661_caspase3_Mesangial', 'assay_type': 'caspase', 'cell_line': 'Mesangial', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '663': {'assay_name': 'AID_663_caspase3_HEK293', 'assay_type': 'caspase', 'cell_line': 'HEK293', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '664': {'assay_name': 'AID_664_caspase3_Jurkat', 'assay_type': 'caspase', 'cell_line': 'Jurkat', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '665': {'assay_name': 'AID_665_caspase3_MRC-5', 'assay_type': 'caspase', 'cell_line': 'MRC-5', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '666': {'assay_name': 'AID_666_caspase3_H-4-II-E', 'assay_type': 'caspase', 'cell_line': 'H-4-II-E', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '667': {'assay_name': 'AID_667_caspase3_RenalProximalTubule', 'assay_type': 'caspase', 'cell_line': 'Renal_Proximal_Tubule', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '588813': {'assay_name': 'AID_588813_caspase3_Lymphoblast', 'assay_type': 'caspase', 'cell_line': 'Lymphoblast', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    # FDA LTKB-BD (Liver Toxicity Knowledge Base Benchmark Dataset)
    '625276': {'assay_name': 'AID_625276_FDA_LTKB_most_concern', 'assay_type': 'DILI_benchmark', 'cell_line': 'clinical', 'source': 'FDA', 'is_counter_screen': False, 'biological_process': 'liver_toxicity', 'endpoint_level': 'clinical', 'dili_concern': 'most_concern'},
    '625277': {'assay_name': 'AID_625277_FDA_LTKB_less_concern', 'assay_type': 'DILI_benchmark', 'cell_line': 'clinical', 'source': 'FDA', 'is_counter_screen': False, 'biological_process': 'liver_toxicity', 'endpoint_level': 'clinical', 'dili_concern': 'less_concern'},
    '625278': {'assay_name': 'AID_625278_FDA_LTKB_no_concern', 'assay_type': 'DILI_benchmark', 'cell_line': 'clinical', 'source': 'FDA', 'is_counter_screen': False, 'biological_process': 'liver_toxicity', 'endpoint_level': 'clinical', 'dili_concern': 'no_concern'}
}

print(f"Defined metadata for {len(ASSAY_METADATA)} assays.")

Defined metadata for 23 assays.


In [3]:
#functions defitions to get the aid name in order to map and skip the rows which starts with result as they are not real data 
def extract_aid_from_filename(filename):
    import re
    match = re.search(r'AID_(\d+)', filename)
    return match.group(1) if match else None

def load_csv_skip_metadata(filepath):
    # First, read just the header to understand structure
    df_peek = pd.read_csv(filepath, nrows=5)
    
    # Check if first column of row 0 starts with 'RESULT_'
    skip_rows = []
    for i in range(min(5, len(df_peek))):
        first_val = str(df_peek.iloc[i, 0])
        if first_val.startswith('RESULT_'):
            skip_rows.append(i + 1)  # +1 because we already have header
    
    # Re-read with appropriate skiprows
    if skip_rows:
        df = pd.read_csv(filepath, skiprows=skip_rows)
    else:
        df = pd.read_csv(filepath)
    
    return df

print("Helper functions defined.")

Helper functions defined.


In [4]:
# Load all datasets
raw_datasets = {} #to store all loaded dataset 
loading_report = [] #to track loading status of each file 

# Scan all subdirectories for CSV files
for subdir in ['mitochondrial', 'tox21_caspase', 'ncats_13_cell_types']:
    subdir_path = BASE_DIR / subdir
    if subdir_path.exists():
        for csv_file in subdir_path.glob('*.csv'):
            aid = extract_aid_from_filename(csv_file.name)
            if aid:
                try:
                    df = load_csv_skip_metadata(csv_file)
                    raw_datasets[aid] = {
                        'data': df,
                        'source_file': str(csv_file),
                        'folder': subdir
                    }
                    loading_report.append({
                        'AID': aid,
                        'File': csv_file.name,
                        'Folder': subdir,
                        'Rows': len(df),
                        'Columns': len(df.columns),
                        'Status': 'Loaded'
                    })
                except Exception as e:
                    loading_report.append({
                        'AID': aid,
                        'File': csv_file.name,
                        'Folder': subdir,
                        'Rows': 0,
                        'Columns': 0,
                        'Status': f'Error: {str(e)[:50]}'
                    })

# Display loading report
loading_df = pd.DataFrame(loading_report)
print("DATA LOADING REPORT")
print(loading_df.to_string(index=False))
print(f"\nTotal datasets loaded: {len(raw_datasets)}")
print(f"Total compounds (sum): {sum(r['Rows'] for r in loading_report):,}")

DATA LOADING REPORT
    AID                                     File              Folder  Rows  Columns Status
 720635                 AID_720635_datatable.csv       mitochondrial 10486     4914 Loaded
 720634       AID_720634_datatable_viability.csv       mitochondrial 10486     1764 Loaded
1346978                AID_1346978_datatable.csv       tox21_caspase  9667     2049 Loaded
1346979 AID_1346979_datatable_counter_screen.csv       tox21_caspase  9667     2049 Loaded
1346981 AID_1346981_datatable_counter_screen.csv       tox21_caspase  9667     2049 Loaded
1346980                AID_1346980_datatable.csv       tox21_caspase  9667     2049 Loaded
    663                    AID_663_datatable.csv ncats_13_cell_types  1408       36 Loaded
    667                    AID_667_datatable.csv ncats_13_cell_types  1408       36 Loaded
 625278                 AID_625278_datatable.csv ncats_13_cell_types    65       10 Loaded
    659                    AID_659_datatable.csv ncats_13_cell_types  

In [5]:
# Inspect each dataset's structure
print("DATASET INSPECTION")

for aid, dataset_info in raw_datasets.items():
    df = dataset_info['data']
    print(f"\n--- AID {aid} ({dataset_info['folder']}) ---")
    print(f"Shape: {df.shape}")
    print(f"\nColumn names ({len(df.columns)} total):")
    # Show first 10 columns
    for i, col in enumerate(df.columns[:10]):
        print(f"  {i+1}. {col}")
    if len(df.columns) > 10:
        print(f"  ... and {len(df.columns) - 10} more columns")
    print()

DATASET INSPECTION

--- AID 720635 (mitochondrial) ---
Shape: (10486, 4914)

Column names (4914 total):
  1. PUBCHEM_RESULT_TAG
  2. PUBCHEM_SID
  3. PUBCHEM_CID
  4. PUBCHEM_EXT_DATASOURCE_SMILES
  5. PUBCHEM_ACTIVITY_OUTCOME
  6. PUBCHEM_ACTIVITY_SCORE
  7. PUBCHEM_ACTIVITY_URL
  8. PUBCHEM_ASSAYDATA_COMMENT
  9. Phenotype-Replicate_1
  10. Potency-Replicate_1
  ... and 4904 more columns


--- AID 720634 (mitochondrial) ---
Shape: (10486, 1764)

Column names (1764 total):
  1. PUBCHEM_RESULT_TAG
  2. PUBCHEM_SID
  3. PUBCHEM_CID
  4. PUBCHEM_EXT_DATASOURCE_SMILES
  5. PUBCHEM_ACTIVITY_OUTCOME
  6. PUBCHEM_ACTIVITY_SCORE
  7. PUBCHEM_ACTIVITY_URL
  8. PUBCHEM_ASSAYDATA_COMMENT
  9. Phenotype-Replicate_1
  10. Potency-Replicate_1
  ... and 1754 more columns


--- AID 1346978 (tox21_caspase) ---
Shape: (9667, 2049)

Column names (2049 total):
  1. PUBCHEM_RESULT_TAG
  2. PUBCHEM_SID
  3. PUBCHEM_CID
  4. PUBCHEM_EXT_DATASOURCE_SMILES
  5. PUBCHEM_ACTIVITY_OUTCOME
  6. PUBCHEM_ACTIVITY_S

In [6]:
# Display sample data from one dataset
sample_aid = list(raw_datasets.keys())[0]
print(f"\nSample data from AID {sample_aid}:")
display(raw_datasets[sample_aid]['data'].head(3))


Sample data from AID 720635:


,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,Phenotype-Replicate_1,Potency-Replicate_1,...,W535-Activity at 0.735 uM-Replicate_45,W535-Activity at 1.635 uM-Replicate_45,W535-Activity at 3.629 uM-Replicate_45,W535-Activity at 8.077 uM-Replicate_45,W535-Activity at 17.94 uM-Replicate_45,W535-Activity at 39.07 uM-Replicate_45,W535-Activity at 83.22 uM-Replicate_45,W535-Activity at 95.27 uM-Replicate_45,W535-Activity at 256.5 uM-Replicate_45,W535-Activity at 920.3 uM-Replicate_45
0,1,144203552,12850184.0,C(C(=O)[C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O.C(...,Inconclusive,10,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Inconclusive,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,144203553,89753.0,C([C@H]([C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O)O...,Inactive,0,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Inactive,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,144203554,9403.0,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2OC(=O)CCC...,Inconclusive,20,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Inhibitor,6.6824,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---

## Remove Unwanted Data

Following the *Remove Unwanted Observations* principle:
- Detect and remove duplicate rows
- Remove rows with missing CID (primary key)
- Drop irrelevant or empty columns
- Log row counts before and after removal

In [7]:
#  REMOVE UNWANTED OBSERVATIONS

def remove_unwanted_observations(df, aid):
    """
    Remove duplicates, missing CIDs, and empty columns.
    """
    report = {
        'aid': aid,
        'initial_rows': len(df),
        'initial_cols': len(df.columns)
    }
    
    # Identify CID column 
    cid_col = None
    for col in ['PUBCHEM_CID', 'CID', 'cid', 'PubChem_CID']:
        if col in df.columns:
            cid_col = col
            break
    
    if cid_col is None:
        report['error'] = 'No CID column found'
        return df, report
    
    report['cid_column'] = cid_col
    
    #  Remove rows with missing CID
    missing_cid_count = df[cid_col].isna().sum()
    df = df.dropna(subset=[cid_col])
    report['missing_cid_removed'] = missing_cid_count
    
    #  Remove duplicate rows (exact duplicates)
    duplicates_count = df.duplicated().sum()
    df = df.drop_duplicates()
    report['duplicates_removed'] = duplicates_count
    
    #  Remove duplicate CIDs (keep first occurrence)
    duplicate_cids = df.duplicated(subset=[cid_col]).sum()
    df = df.drop_duplicates(subset=[cid_col], keep='first')
    report['duplicate_cids_removed'] = duplicate_cids
    
    # Remove columns that are completely empty
    empty_cols = df.columns[df.isna().all()].tolist()
    df = df.dropna(axis=1, how='all')
    report['empty_cols_removed'] = len(empty_cols)
    report['empty_col_names'] = empty_cols[:5] if empty_cols else []
    
    #  Remove columns with >95% missing values (likely irrelevant)
    high_missing_cols = df.columns[df.isna().mean() > 0.95].tolist()
    report['high_missing_cols'] = len(high_missing_cols)
    
    report['final_rows'] = len(df)
    report['final_cols'] = len(df.columns)
    report['rows_removed_pct'] = round((1 - len(df)/report['initial_rows']) * 100, 2) if report['initial_rows'] > 0 else 0
    
    return df, report

print("Observation removal function defined.")

Observation removal function defined.


In [8]:
# Apply observation removal to all datasets
cleaned_datasets = {} #will store cleaned dataframes
removal_reports = [] #will store cleaning reports for each dataset

for aid, dataset_info in raw_datasets.items():
    df_cleaned, report = remove_unwanted_observations(
        dataset_info['data'].copy(), #to prevent loss of original dataset
        aid
    )
    cleaned_datasets[aid] = {
        'data': df_cleaned,
        'source_file': dataset_info['source_file'],
        'folder': dataset_info['folder']
    }
    removal_reports.append(report)

# Display removal report
print("OBSERVATION REMOVAL REPORT")

removal_df = pd.DataFrame(removal_reports)
display_cols = ['aid', 'initial_rows', 'missing_cid_removed', 'duplicates_removed', 
                'duplicate_cids_removed', 'final_rows', 'rows_removed_pct']
print(removal_df[display_cols].to_string(index=False))

print(f"\nTotal rows before cleaning: {sum(r['initial_rows'] for r in removal_reports):,}")
print(f"Total rows after cleaning: {sum(r['final_rows'] for r in removal_reports):,}")

OBSERVATION REMOVAL REPORT
    aid  initial_rows  missing_cid_removed  duplicates_removed  duplicate_cids_removed  final_rows  rows_removed_pct
 720635         10486                  149                   0                    2238        8099             22.76
 720634         10486                  149                   0                    2238        8099             22.76
1346978          9667                  143                   0                    1853        7671             20.65
1346979          9667                  143                   0                    1853        7671             20.65
1346981          9667                  143                   0                    1853        7671             20.65
1346980          9667                  143                   0                    1853        7671             20.65
    663          1408                   14                   0                      59        1335              5.18
    667          1408                

---

## Fix Errors

Following *Fix Structural Errors* principle:
- Standardize column names across all datasets
- Normalize outcome labels (Active/Inactive/Inconclusive)
- Ensure correct data types
- Enforce one SMILES per CID, flag inconsistencies

In [9]:
# FIX STRUCTURAL ERRORS

# Define standard column mapping
COLUMN_MAPPING = {
    # CID variations
    'PUBCHEM_CID': 'CID',
    'cid': 'CID',
    'PubChem_CID': 'CID',
    'pubchem_cid': 'CID',
    
    # SMILES variations
    'PUBCHEM_EXT_DATASOURCE_SMILES': 'SMILES',
    'smiles': 'SMILES',
    'Smiles': 'SMILES',
    'CANONICAL_SMILES': 'SMILES',
    
    # Outcome variations
    'PUBCHEM_ACTIVITY_OUTCOME': 'outcome',
    'Activity_Outcome': 'outcome',
    'activity_outcome': 'outcome',
    
    # Score variations
    'PUBCHEM_ACTIVITY_SCORE': 'score',
    'Activity_Score': 'score',
    'activity_score': 'score'
}

# Define standard outcome labels
#PubChem activity outcome values classify chemical test results as
# 1 (inactive), 2 (active), 3 (inconclusive), 4 (unspecified), or 5 (probe), allowing for standardized filtering of biological screening data.
OUTCOME_MAPPING = {
    'Active': 'Active',
    'active': 'Active',
    'ACTIVE': 'Active',
    'Inactive': 'Inactive',
    'inactive': 'Inactive',
    'INACTIVE': 'Inactive',
    'Inconclusive': 'Inconclusive',
    'inconclusive': 'Inconclusive',
    'INCONCLUSIVE': 'Inconclusive',
    'Unspecified': 'Inconclusive',
    'unspecified': 'Inconclusive',
    'Probe': 'Active' 
}

print("Column and outcome mappings defined.")

Column and outcome mappings defined.


In [10]:
def fix_structural_errors(df, aid):
    """
    Standardize column names, outcome labels, and data types.
    """
    report = {'aid': aid}
    df = df.copy()
    
    #  Standardize column names
    columns_renamed = []
    for old_name, new_name in COLUMN_MAPPING.items():
        if old_name in df.columns and old_name != new_name:
            df = df.rename(columns={old_name: new_name})
            columns_renamed.append(f"{old_name} -> {new_name}")
    report['columns_renamed'] = len(columns_renamed)
    
    #  Normalize outcome labels
    if 'outcome' in df.columns:
        original_outcomes = df['outcome'].unique().tolist()
        df['outcome'] = df['outcome'].map(lambda x: OUTCOME_MAPPING.get(str(x), 'Inconclusive'))
        report['original_outcomes'] = original_outcomes
        report['standardized_outcomes'] = df['outcome'].unique().tolist()
    
    #  Ensure CID is integer type
    if 'CID' in df.columns:
        # Convert to numeric, coercing errors to NaN
        df['CID'] = pd.to_numeric(df['CID'], errors='coerce')
        # Drop rows where CID couldn't be converted
        invalid_cids = df['CID'].isna().sum()
        df = df.dropna(subset=['CID'])
        df['CID'] = df['CID'].astype(int)
        report['invalid_cids_removed'] = invalid_cids
    
    #  Ensure score is numeric
    if 'score' in df.columns:
        df['score'] = pd.to_numeric(df['score'], errors='coerce')
        report['score_type'] = str(df['score'].dtype)
    
    report['final_rows'] = len(df)
    return df, report

print("Structural error fixing function defined.")

Structural error fixing function defined.


In [11]:
# Apply structural fixes to all datasets
structural_reports = []

for aid in cleaned_datasets:
    df_fixed, report = fix_structural_errors(
        cleaned_datasets[aid]['data'], 
        aid
    )
    cleaned_datasets[aid]['data'] = df_fixed
    structural_reports.append(report)

# Display structural fix report
print("\n" + "="*70)
print("STRUCTURAL ERRORS FIX REPORT")
print("="*70)

for report in structural_reports:
    print(f"\nAID {report['aid']}:")
    print(f"  Columns renamed: {report.get('columns_renamed', 0)}")
    if 'original_outcomes' in report:
        print(f"  Original outcomes: {report['original_outcomes']}")
        print(f"  Standardized outcomes: {report['standardized_outcomes']}")
    if 'invalid_cids_removed' in report:
        print(f"  Invalid CIDs removed: {report['invalid_cids_removed']}")


STRUCTURAL ERRORS FIX REPORT

AID 720635:
  Columns renamed: 4
  Original outcomes: ['Inconclusive', 'Inactive', 'Active']
  Standardized outcomes: ['Inconclusive', 'Inactive', 'Active']
  Invalid CIDs removed: 0

AID 720634:
  Columns renamed: 4
  Original outcomes: ['Inactive', 'Inconclusive', 'Active']
  Standardized outcomes: ['Inactive', 'Inconclusive', 'Active']
  Invalid CIDs removed: 0

AID 1346978:
  Columns renamed: 4
  Original outcomes: ['Inactive', 'Inconclusive', 'Active']
  Standardized outcomes: ['Inactive', 'Inconclusive', 'Active']
  Invalid CIDs removed: 0

AID 1346979:
  Columns renamed: 4
  Original outcomes: ['Inactive', 'Active', 'Inconclusive']
  Standardized outcomes: ['Inactive', 'Active', 'Inconclusive']
  Invalid CIDs removed: 0

AID 1346981:
  Columns renamed: 4
  Original outcomes: ['Inactive', 'Inconclusive', 'Active']
  Standardized outcomes: ['Inactive', 'Inconclusive', 'Active']
  Invalid CIDs removed: 0

AID 1346980:
  Columns renamed: 4
  Original o

In [12]:
# Check SMILES consistency per CID
print("SMILES CONSISTENCY CHECK")

# Collect all CID-SMILES pairs
cid_smiles_pairs = []
for aid, dataset_info in cleaned_datasets.items():
    df = dataset_info['data']
    if 'CID' in df.columns and 'SMILES' in df.columns:
        pairs = df[['CID', 'SMILES']].dropna().drop_duplicates()
        cid_smiles_pairs.append(pairs)

if cid_smiles_pairs:
    all_pairs = pd.concat(cid_smiles_pairs, ignore_index=True)
    
    # Find CIDs with multiple different SMILES
    smiles_per_cid = all_pairs.groupby('CID')['SMILES'].nunique()
    inconsistent_cids = smiles_per_cid[smiles_per_cid > 1]
    
    print(f"Total unique CIDs: {len(smiles_per_cid):,}")
    print(f"CIDs with consistent SMILES: {(smiles_per_cid == 1).sum():,}")
    print(f"CIDs with inconsistent SMILES: {len(inconsistent_cids):,}")
    
    if len(inconsistent_cids) > 0:
        print(f"\nWarning: {len(inconsistent_cids)} CIDs have multiple SMILES representations.")
        print("This may indicate tautomers, stereoisomers, or data entry errors.")
        print("\nExample inconsistent CIDs:")
        for cid in inconsistent_cids.head(3).index:
            smiles_list = all_pairs[all_pairs['CID'] == cid]['SMILES'].unique()
            print(f"  CID {cid}: {len(smiles_list)} different SMILES")

SMILES CONSISTENCY CHECK
Total unique CIDs: 8,882
CIDs with consistent SMILES: 8,882
CIDs with inconsistent SMILES: 0


---

## Identify Column Data Types

Explicitly separate categorical and numerical columns for proper handling.

In [13]:

# IDENTIFY COLUMN DATA TYPES

def categorize_columns(df):
    """
    Categorize columns into numerical and categorical types.
    """
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    boolean_cols = df.select_dtypes(include=['bool']).columns.tolist()
    
    return {
        'numerical': numerical_cols,
        'categorical': categorical_cols,
        'boolean': boolean_cols
    }

# Analyze column types for each dataset
print("COLUMN DATA TYPE ANALYSIS")

for aid in list(cleaned_datasets.keys())[:3]:  # Show first 3 as examples
    df = cleaned_datasets[aid]['data']
    col_types = categorize_columns(df)
    
    print(f"\nAID {aid}:")
    print(f"  Numerical columns ({len(col_types['numerical'])}): ", end="")
    print(", ".join(col_types['numerical'][:5]) + ("..." if len(col_types['numerical']) > 5 else ""))
    print(f"  Categorical columns ({len(col_types['categorical'])}): ", end="")
    print(", ".join(col_types['categorical'][:5]) + ("..." if len(col_types['categorical']) > 5 else ""))

COLUMN DATA TYPE ANALYSIS

AID 720635:
  Numerical columns (3541): PUBCHEM_RESULT_TAG, PUBCHEM_SID, CID, score, Potency-Replicate_1...
  Categorical columns (316): SMILES, outcome, PUBCHEM_ACTIVITY_URL, Phenotype-Replicate_1, W590-Curve_Description-Replicate_1...

AID 720634:
  Numerical columns (1237): PUBCHEM_RESULT_TAG, PUBCHEM_SID, CID, score, Potency-Replicate_1...
  Categorical columns (136): SMILES, outcome, PUBCHEM_ACTIVITY_URL, Phenotype-Replicate_1, Curve_Description-Replicate_1...

AID 1346978:
  Numerical columns (1243): PUBCHEM_RESULT_TAG, PUBCHEM_SID, CID, score, Potency-Replicate_1...
  Categorical columns (154): SMILES, outcome, PUBCHEM_ACTIVITY_URL, Phenotype-Replicate_1, Curve_Description-Replicate_1...


In [14]:
# Define core columns for the standardized output
CORE_COLUMNS = [
    'CID',           # Primary key
    'SMILES',        # Molecular structure
    'outcome',       # Activity classification
    'score'          # Numeric activity score
]

# Potency-related columns to extract if available
POTENCY_COLUMNS = [
    'Potency',
    'Potency-Replicate_1',
    'Potency-Replicate_2',
    'Efficacy',
    'Efficacy-Replicate_1',
    'Fit_LogAC50',
    'Fit_LogAC50-Replicate_1'
]

print("Core and potency columns defined.")
print(f"\nCore columns: {CORE_COLUMNS}")
print(f"Potency columns to extract (if available): {POTENCY_COLUMNS[:4]}...")

Core and potency columns defined.

Core columns: ['CID', 'SMILES', 'outcome', 'score']
Potency columns to extract (if available): ['Potency', 'Potency-Replicate_1', 'Potency-Replicate_2', 'Efficacy']...


---

##Handle Missing Data

Following *Handle Missing Data* principle:
- Report missing value percentages
- Apply appropriate imputation 

In [15]:

#  HANDLE MISSING DATA

def analyze_missing_data(df, aid):
    """
    Analyze and report missing data patterns.
    """
    total_cells = df.size
    missing_cells = df.isna().sum().sum()
    
    report = {
        'aid': aid,
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'total_cells': total_cells,
        'missing_cells': missing_cells,
        'missing_pct': round(missing_cells / total_cells * 100, 2) if total_cells > 0 else 0
    }
    
    # Analyze core columns specifically
    for col in CORE_COLUMNS:
        if col in df.columns:
            missing = df[col].isna().sum()
            pct = round(missing / len(df) * 100, 2) if len(df) > 0 else 0
            report[f'{col}_missing'] = missing
            report[f'{col}_missing_pct'] = pct
    
    return report

# Analyze missing data for all datasets
missing_reports = []
for aid, dataset_info in cleaned_datasets.items():
    report = analyze_missing_data(dataset_info['data'], aid)
    missing_reports.append(report)

# Display missing data report
print("MISSING DATA ANALYSIS")

missing_df = pd.DataFrame(missing_reports)
core_missing_cols = ['aid', 'total_rows', 'missing_pct']
for col in CORE_COLUMNS:
    if f'{col}_missing_pct' in missing_df.columns:
        core_missing_cols.append(f'{col}_missing_pct')

print(missing_df[core_missing_cols].to_string(index=False))

MISSING DATA ANALYSIS
    aid  total_rows  missing_pct  CID_missing_pct  SMILES_missing_pct  outcome_missing_pct  score_missing_pct
 720635        8099        90.98              0.0                 0.0                  0.0                0.0
 720634        8099        91.15              0.0                 0.0                  0.0                0.0
1346978        7671        90.95              0.0                 0.0                  0.0                0.0
1346979        7671        90.56              0.0                 0.0                  0.0                0.0
1346981        7671        90.76              0.0                 0.0                  0.0                0.0
1346980        7671        91.03              0.0                 0.0                  0.0                0.0
    663        1335        26.76              0.0                 0.0                  0.0                0.0
    667        1335        27.43              0.0                 0.0                  0.0        

In [16]:
def handle_missing_values(df, aid, strategy='conservative'):
    #handle missing values
    df = df.copy()
    report = {'aid': aid, 'imputation_actions': []}
    
    # CID: Cannot impute - remove if missing
    if 'CID' in df.columns:
        missing_cid = df['CID'].isna().sum()
        if missing_cid > 0:
            df = df.dropna(subset=['CID'])
            report['imputation_actions'].append(f"Removed {missing_cid} rows with missing CID")
    
    # SMILES: Cannot impute - flag but keep (can retrieve later via CID)
    if 'SMILES' in df.columns:
        missing_smiles = df['SMILES'].isna().sum()
        if missing_smiles > 0:
            report['imputation_actions'].append(f"Flagged {missing_smiles} rows with missing SMILES (can retrieve via CID)")
    
    # Outcome: Impute as 'Inconclusive' if missing
    if 'outcome' in df.columns:
        missing_outcome = df['outcome'].isna().sum()
        if missing_outcome > 0:
            df['outcome'] = df['outcome'].fillna('Inconclusive')
            report['imputation_actions'].append(f"Imputed {missing_outcome} missing outcomes as 'Inconclusive'")
    
    # Score: Impute with median 
    if 'score' in df.columns:
        missing_score = df['score'].isna().sum()
        if missing_score > 0 and strategy == 'aggressive':
            median_score = df['score'].median()
            df['score'] = df['score'].fillna(median_score)
            report['imputation_actions'].append(f"Imputed {missing_score} missing scores with median ({median_score:.2f})")
        elif missing_score > 0:
            report['imputation_actions'].append(f"Flagged {missing_score} missing scores (not imputed - conservative)")
    
    report['final_rows'] = len(df)
    return df, report

print("Missing value handling function defined.")

Missing value handling function defined.


In [17]:
# Apply missing value handling
imputation_reports = []

for aid in cleaned_datasets:
    df_imputed, report = handle_missing_values(
        cleaned_datasets[aid]['data'],
        aid,
        strategy='conservative'  # Use conservative approach
    )
    cleaned_datasets[aid]['data'] = df_imputed
    imputation_reports.append(report)

for report in imputation_reports:
    if report['imputation_actions']:
        print(f"\nAID {report['aid']}:")
        for action in report['imputation_actions']:
            print(f"  - {action}")

---

## Manage Outliers

Following *Manage Outliers* principle:
- Identify numeric outliers using IQR method
- Flag but do NOT blindly remove

In [19]:

#MANAGE OUTLIERS

def detect_outliers_iqr(series, multiplier=1.5):
    """
    Detect outliers using the IQR method.
    Returns boolean mask where True = outlier.
    """
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    
    outliers = (series < lower_bound) | (series > upper_bound)
    
    return outliers, {'Q1': Q1, 'Q3': Q3, 'IQR': IQR, 
                      'lower_bound': lower_bound, 'upper_bound': upper_bound}

def analyze_outliers(df, aid):
    """
    Analyze outliers in numerical columns.
    """
    report = {'aid': aid, 'outlier_analysis': []}
    
    # Check 'score' column
    if 'score' in df.columns:
        score_data = df['score'].dropna()
        if len(score_data) > 0:
            outliers, stats = detect_outliers_iqr(score_data)
            n_outliers = outliers.sum()
            pct_outliers = round(n_outliers / len(score_data) * 100, 2)
            
            report['outlier_analysis'].append({
                'column': 'score',
                'n_outliers': n_outliers,
                'pct_outliers': pct_outliers,
                'lower_bound': round(stats['lower_bound'], 2),
                'upper_bound': round(stats['upper_bound'], 2),
                'min_value': round(score_data.min(), 2),
                'max_value': round(score_data.max(), 2)
            })
    
    return report

print("Outlier detection functions defined.")

Outlier detection functions defined.


In [20]:
# Analyze outliers in all datasets
outlier_reports = []

for aid, dataset_info in cleaned_datasets.items():
    report = analyze_outliers(dataset_info['data'], aid)
    outlier_reports.append(report)

# Display outlier analysis
print("\n" + "="*70)
print("OUTLIER ANALYSIS REPORT")
print("="*70)

outlier_summary = []
for report in outlier_reports:
    for analysis in report['outlier_analysis']:
        outlier_summary.append({
            'AID': report['aid'],
            'Column': analysis['column'],
            'N_Outliers': analysis['n_outliers'],
            'Pct': analysis['pct_outliers'],
            'Range': f"[{analysis['lower_bound']}, {analysis['upper_bound']}]",
            'Actual': f"[{analysis['min_value']}, {analysis['max_value']}]"
        })

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary)
    print(outlier_df.to_string(index=False))


OUTLIER ANALYSIS REPORT
    AID Column  N_Outliers   Pct         Range   Actual
 720635  score        1297 16.01 [-15.0, 25.0]  [0, 99]
 720634  score         840 10.37    [0.0, 0.0]  [0, 88]
1346978  score         499  6.51    [0.0, 0.0]  [0, 94]
1346979  score         335  4.37 [-30.0, 50.0]  [0, 97]
1346981  score        1386 18.07    [0.0, 0.0]  [0, 92]
1346980  score         333  4.34    [0.0, 0.0]  [0, 90]
    663  score          65  4.87    [0.0, 0.0] [0, 100]
    667  score          33  2.47    [0.0, 0.0] [0, 100]
    659  score          30  2.25    [0.0, 0.0] [0, 100]
    660  score          35  2.62    [0.0, 0.0] [0, 100]
    664  score          58  4.34    [0.0, 0.0] [0, 100]
    657  score          46  3.45    [0.0, 0.0] [0, 100]
    654  score          37  2.77    [0.0, 0.0] [0, 100]
    655  score         130  9.74    [0.0, 0.0] [0, 100]
    656  score          20  1.50    [0.0, 0.0] [0, 100]
    665  score          28  2.10    [0.0, 0.0] [0, 100]
    661  score         

---

##Assay Annotation

Add explicit toxicology-specific metadata columns:
- `is_counter_screen` (True/False)
- `biological_process` (mitochondria / apoptosis / cytotoxicity)
- `endpoint_level` (mechanistic vs general toxicity)

In [21]:

# ASSAY ANNOTATION (DOMAIN-AWARE)

def annotate_assay(df, aid):
    """
    Add toxicology-specific metadata columns based on assay type.
    """
    df = df.copy()
    
    # Get metadata for this assay
    if aid in ASSAY_METADATA:
        metadata = ASSAY_METADATA[aid]
        
        df['assay_name'] = metadata['assay_name']
        df['assay_type'] = metadata['assay_type']
        df['cell_line'] = metadata['cell_line']
        df['source'] = metadata['source']
        df['is_counter_screen'] = metadata['is_counter_screen']
        df['biological_process'] = metadata['biological_process']
        df['endpoint_level'] = metadata['endpoint_level']
    else:
        # Default values for unknown assays
        df['assay_name'] = f'AID_{aid}'
        df['assay_type'] = 'unknown'
        df['cell_line'] = 'unknown'
        df['source'] = 'unknown'
        df['is_counter_screen'] = False
        df['biological_process'] = 'unknown'
        df['endpoint_level'] = 'unknown'
    
    return df

# Apply annotations to all datasets
for aid in cleaned_datasets:
    cleaned_datasets[aid]['data'] = annotate_assay(
        cleaned_datasets[aid]['data'], 
        aid
    )

print("Assay annotations applied.")

Assay annotations applied.


In [22]:
# Display annotation summary
print("\n" + "="*70)
print("ASSAY ANNOTATION SUMMARY")
print("="*70)

annotation_summary = []
for aid, dataset_info in cleaned_datasets.items():
    df = dataset_info['data']
    if 'assay_name' in df.columns:
        annotation_summary.append({
            'AID': aid,
            'Assay_Name': df['assay_name'].iloc[0] if len(df) > 0 else 'N/A',
            'Type': df['assay_type'].iloc[0] if len(df) > 0 else 'N/A',
            'Cell_Line': df['cell_line'].iloc[0] if len(df) > 0 else 'N/A',
            'Source': df['source'].iloc[0] if len(df) > 0 else 'N/A',
            'Counter_Screen': df['is_counter_screen'].iloc[0] if len(df) > 0 else 'N/A',
            'Process': df['biological_process'].iloc[0] if len(df) > 0 else 'N/A',
            'N_Compounds': len(df)
        })

annotation_df = pd.DataFrame(annotation_summary)
print(annotation_df.to_string(index=False))


ASSAY ANNOTATION SUMMARY
    AID                           Assay_Name           Type             Cell_Line Source  Counter_Screen        Process  N_Compounds
 720635   AID_720635_mito_membrane_potential  mitochondrial                 HepG2  Tox21           False   mitochondria         8099
 720634            AID_720634_cell_viability      viability                 HepG2  Tox21            True   cytotoxicity         8099
1346978          AID_1346978_caspase37_HepG2        caspase                 HepG2  Tox21           False      apoptosis         7671
1346979         AID_1346979_viability_CHO-K1      viability                CHO-K1  Tox21            True   cytotoxicity         7671
1346981          AID_1346981_viability_HepG2      viability                 HepG2  Tox21            True   cytotoxicity         7671
1346980         AID_1346980_caspase37_CHO-K1        caspase                CHO-K1  Tox21           False      apoptosis         7671
    663              AID_663_caspase3_HEK29

---

## Merging

ML-safe merging:
- Merge datasets using **CID as the join key**

In [23]:

#  MERGING STRATEGY

# Define the standardized columns for the merged dataset
STANDARD_COLUMNS = [
    'CID',
    'SMILES',
    'assay_name',
    'assay_type',
    'cell_line',
    'outcome',
    'score',
    'source',
    'is_counter_screen',
    'biological_process',
    'endpoint_level'
]

def extract_standard_columns(df, aid):
    """
    Extract and standardize columns for merging.
    """
    available_cols = [col for col in STANDARD_COLUMNS if col in df.columns]
    df_standard = df[available_cols].copy()
    
    # Ensure all standard columns exist
    for col in STANDARD_COLUMNS:
        if col not in df_standard.columns:
            df_standard[col] = np.nan
    
    return df_standard[STANDARD_COLUMNS]

print(f"Standard columns for merging: {STANDARD_COLUMNS}")

Standard columns for merging: ['CID', 'SMILES', 'assay_name', 'assay_type', 'cell_line', 'outcome', 'score', 'source', 'is_counter_screen', 'biological_process', 'endpoint_level']


In [24]:
# Extract standard columns from all datasets and concatenate
standardized_dfs = []

for aid, dataset_info in cleaned_datasets.items():
    df_standard = extract_standard_columns(dataset_info['data'], aid)
    standardized_dfs.append(df_standard)
    print(f"AID {aid}: {len(df_standard)} rows extracted")

# Concatenate all datasets (long format - preserves all assay results)
merged_long = pd.concat(standardized_dfs, ignore_index=True)

print(f"\n" + "="*70)
print("MERGED DATASET (LONG FORMAT)")
print("="*70)
print(f"Total rows: {len(merged_long):,}")
print(f"Unique CIDs: {merged_long['CID'].nunique():,}")
print(f"Unique assays: {merged_long['assay_name'].nunique()}")
print(f"\nShape: {merged_long.shape}")

AID 720635: 8099 rows extracted
AID 720634: 8099 rows extracted
AID 1346978: 7671 rows extracted
AID 1346979: 7671 rows extracted
AID 1346981: 7671 rows extracted
AID 1346980: 7671 rows extracted
AID 663: 1335 rows extracted
AID 667: 1335 rows extracted
AID 625278: 61 rows extracted
AID 659: 1335 rows extracted
AID 660: 1335 rows extracted
AID 664: 1335 rows extracted
AID 657: 1335 rows extracted
AID 654: 1335 rows extracted
AID 625276: 135 rows extracted
AID 625277: 83 rows extracted
AID 655: 1335 rows extracted
AID 656: 1335 rows extracted
AID 665: 1335 rows extracted
AID 661: 1335 rows extracted
AID 658: 1335 rows extracted
AID 666: 1335 rows extracted

MERGED DATASET (LONG FORMAT)
Total rows: 64,516
Unique CIDs: 8,882
Unique assays: 22

Shape: (64516, 11)


In [25]:
# Display sample of merged data
print("\nSample of merged long-format data:")
display(merged_long.head(10))


Sample of merged long-format data:


,CID,SMILES,assay_name,assay_type,cell_line,outcome,score,source,is_counter_screen,biological_process,endpoint_level
0,12850184,C(C(=O)[C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O.C(...,AID_720635_mito_membrane_potential,mitochondrial,HepG2,Inconclusive,10.0,Tox21,False,mitochondria,mechanistic
1,89753,C([C@H]([C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O)O...,AID_720635_mito_membrane_potential,mitochondrial,HepG2,Inactive,0.0,Tox21,False,mitochondria,mechanistic
2,9403,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2OC(=O)CCC...,AID_720635_mito_membrane_potential,mitochondrial,HepG2,Inconclusive,20.0,Tox21,False,mitochondria,mechanistic
3,13218779,C[C@@]12CC[C@@H](C1(C)C)C[C@H]2OC(=O)CSC#N,AID_720635_mito_membrane_potential,mitochondrial,HepG2,Inconclusive,10.0,Tox21,False,mitochondria,mechanistic
4,142766,C1=CC=C(C(=C1)C(=O)O)O.C1=CC2=C(C(=C1)O)N=CC=C2,AID_720635_mito_membrane_potential,mitochondrial,HepG2,Active,41.0,Tox21,False,mitochondria,mechanistic
5,16043,CC(C)(C)C1=C(C=CC(=C1)O)O,AID_720635_mito_membrane_potential,mitochondrial,HepG2,Inactive,0.0,Tox21,False,mitochondria,mechanistic
6,2724411,CN(C)C1=CC=C(C=C1)C(=C2C=CC(=[N+](C)C)C=C2)C3=...,AID_720635_mito_membrane_potential,mitochondrial,HepG2,Active,40.0,Tox21,False,mitochondria,mechanistic
7,90454,C=C[C@H]1CN2CC[C@H]1C[C@@H]2[C@H](C3=CC=NC4=CC...,AID_720635_mito_membrane_potential,mitochondrial,HepG2,Active,41.0,Tox21,False,mitochondria,mechanistic
8,637566,CC(=CCC/C(=C/CO)/C)C,AID_720635_mito_membrane_potential,mitochondrial,HepG2,Inconclusive,10.0,Tox21,False,mitochondria,mechanistic
9,994,C1=CC=C(C=C1)CC(C(=O)O)N,AID_720635_mito_membrane_potential,mitochondrial,HepG2,Inactive,0.0,Tox21,False,mitochondria,mechanistic


In [26]:
# Resolve SMILES inconsistencies: Keep first valid SMILES per CID
cid_smiles_map = merged_long.dropna(subset=['SMILES']).groupby('CID')['SMILES'].first().to_dict()

# Fill missing SMILES using the map
merged_long['SMILES'] = merged_long.apply(
    lambda row: cid_smiles_map.get(row['CID'], row['SMILES']) if pd.isna(row['SMILES']) else row['SMILES'],
    axis=1
)

print(f"SMILES filled from CID mapping.")
print(f"Remaining missing SMILES: {merged_long['SMILES'].isna().sum()}")

SMILES filled from CID mapping.
Remaining missing SMILES: 0


---

## Data Validation & Verification

Following *Data validation and verification*:
- Number of unique CIDs
- Assays per CID distribution
- Outcome class balance
- Missing value summary
- Biological sanity checks

In [27]:

#  DATA VALIDATION & VERIFICATION

print("\n" + "="*70)
print("DATA VALIDATION REPORT")
print("="*70)

# 9.1 Basic statistics
print("\n--- 9.1 BASIC STATISTICS ---")
print(f"Total records: {len(merged_long):,}")
print(f"Unique CIDs: {merged_long['CID'].nunique():,}")
print(f"Unique SMILES: {merged_long['SMILES'].nunique():,}")
print(f"Unique assays: {merged_long['assay_name'].nunique()}")


DATA VALIDATION REPORT

--- 9.1 BASIC STATISTICS ---
Total records: 64,516
Unique CIDs: 8,882
Unique SMILES: 8,880
Unique assays: 22


In [28]:
# 9.2 Assays per CID distribution
print("\n--- 9.2 ASSAYS PER CID DISTRIBUTION ---")
assays_per_cid = merged_long.groupby('CID')['assay_name'].nunique()

print(f"Min assays per CID: {assays_per_cid.min()}")
print(f"Max assays per CID: {assays_per_cid.max()}")
print(f"Mean assays per CID: {assays_per_cid.mean():.2f}")
print(f"Median assays per CID: {assays_per_cid.median():.1f}")

# Distribution
print("\nDistribution of assay coverage:")
coverage_bins = pd.cut(assays_per_cid, bins=[0, 1, 5, 10, 15, 20], labels=['1', '2-5', '6-10', '11-15', '16-20'])
print(coverage_bins.value_counts().sort_index())


--- 9.2 ASSAYS PER CID DISTRIBUTION ---
Min assays per CID: 1
Max assays per CID: 20
Mean assays per CID: 7.26
Median assays per CID: 6.0

Distribution of assay coverage:
assay_name
1         105
2-5      1695
6-10     5747
11-15      45
16-20    1290
Name: count, dtype: int64


In [29]:
# 9.3 Outcome class balance
print("\n--- 9.3 OUTCOME CLASS BALANCE ---")

# Overall
print("\nOverall outcome distribution:")
outcome_counts = merged_long['outcome'].value_counts()
outcome_pcts = merged_long['outcome'].value_counts(normalize=True) * 100
for outcome in outcome_counts.index:
    print(f"  {outcome}: {outcome_counts[outcome]:,} ({outcome_pcts[outcome]:.1f}%)")

# By assay type
print("\nOutcome distribution by assay type:")
outcome_by_type = merged_long.groupby(['assay_type', 'outcome']).size().unstack(fill_value=0)
print(outcome_by_type)


--- 9.3 OUTCOME CLASS BALANCE ---

Overall outcome distribution:
  Inactive: 53,869 (83.5%)
  Inconclusive: 6,892 (10.7%)
  Active: 3,755 (5.8%)

Outcome distribution by assay type:
outcome         Active  Inactive  Inconclusive
assay_type                                    
DILI_benchmark       0         0           279
caspase            507     31234           956
mitochondrial     1297      3565          3237
viability         1951     19070          2420


In [30]:
# 9.4 Missing value summary 
print("\n--- 9.4 FINAL MISSING VALUE SUMMARY ---")

missing_summary = merged_long.isna().sum()
missing_pct = (merged_long.isna().sum() / len(merged_long) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing_Count': missing_summary,
    'Missing_Pct': missing_pct
})
print(missing_report[missing_report['Missing_Count'] > 0])


--- 9.4 FINAL MISSING VALUE SUMMARY ---
       Missing_Count  Missing_Pct
score            279         0.43


In [31]:
# 9.5 Biological 
print("\n--- 9.5 BIOLOGICAL SANITY CHECKS ---")

# Check: Counter-screens should have more inactives than primary assays
print("\nCounter-screen vs Primary assay activity rates:")
activity_by_screen = merged_long.groupby('is_counter_screen').apply(
    lambda x: (x['outcome'] == 'Active').mean() * 100
)
print(f"  Primary assays (is_counter_screen=False): {activity_by_screen.get(False, 0):.2f}% active")
print(f"  Counter-screens (is_counter_screen=True): {activity_by_screen.get(True, 0):.2f}% active")

# Check: Cell line coverage
print("\nCell line coverage:")
cell_line_counts = merged_long.groupby('cell_line')['CID'].nunique().sort_values(ascending=False)
for cell_line, count in cell_line_counts.head(10).items():
    print(f"  {cell_line}: {count:,} unique compounds")

# Check: Source distribution
print("\nSource distribution:")
source_counts = merged_long['source'].value_counts()
for source, count in source_counts.items():
    print(f"  {source}: {count:,} records")


--- 9.5 BIOLOGICAL SANITY CHECKS ---

Counter-screen vs Primary assay activity rates:
  Primary assays (is_counter_screen=False): 4.39% active
  Counter-screens (is_counter_screen=True): 8.32% active

Cell line coverage:
  HepG2: 8,777 unique compounds
  CHO-K1: 7,671 unique compounds
  BJ: 1,335 unique compounds
  H-4-II-E: 1,335 unique compounds
  HEK293: 1,335 unique compounds
  HUV-EC-C: 1,335 unique compounds
  Jurkat: 1,335 unique compounds
  MRC-5: 1,335 unique compounds
  Mesangial: 1,335 unique compounds
  N2a: 1,335 unique compounds

Source distribution:
  Tox21: 46,882 records
  NCATS: 17,355 records
  FDA: 279 records


In [32]:
# 9.6 Cross-validation: Viability vs Caspase correlation
print("\n--- 9.6 BIOLOGICAL CROSS-VALIDATION ---")
print("\nExpected: Compounds active in caspase assays should often show")
print("reduced viability (cytotoxicity) in counter-screens.")

# Find compounds active in caspase assays
caspase_active = merged_long[
    (merged_long['assay_type'] == 'caspase') & 
    (merged_long['outcome'] == 'Active')
]['CID'].unique()

# Check their viability outcomes
viability_of_caspase_active = merged_long[
    (merged_long['CID'].isin(caspase_active)) & 
    (merged_long['assay_type'] == 'viability')
]['outcome'].value_counts()

print(f"\nCompounds active in caspase assays: {len(caspase_active):,}")
print("Their viability outcomes:")
for outcome, count in viability_of_caspase_active.items():
    print(f"  {outcome}: {count:,}")


--- 9.6 BIOLOGICAL CROSS-VALIDATION ---

Expected: Compounds active in caspase assays should often show
reduced viability (cytotoxicity) in counter-screens.

Compounds active in caspase assays: 318
Their viability outcomes:
  Active: 476
  Inactive: 272
  Inconclusive: 177


---

## Final Output

Export:
- `cleaned_long_format.csv` - Full dataset in long format
- Summary statistics and data dictionary

In [33]:

#  FINAL OUTPUT

# Create output directory
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

#  Export long format
long_output_path = OUTPUT_DIR / 'cleaned_long_format.csv'
merged_long.to_csv(long_output_path, index=False)
print(f"Long format saved to: {long_output_path}")
print(f"  Rows: {len(merged_long):,}")
print(f"  Columns: {len(merged_long.columns)}")

Long format saved to: output/cleaned_long_format.csv
  Rows: 64,516
  Columns: 11


In [34]:
# Create data dictionary
data_dictionary = {
    'CID': 'PubChem Compound ID - unique identifier for each molecule',
    'SMILES': 'Canonical SMILES string representing molecular structure',
    'assay_name': 'Unique identifier for the bioassay (AID_xxx_description)',
    'assay_type': 'Type of assay: caspase, mitochondrial, or viability',
    'cell_line': 'Cell line used in the assay',
    'outcome': 'Activity classification: Active, Inactive, or Inconclusive',
    'score': 'Numeric activity score from PubChem (higher = more confident activity)',
    'source': 'Data source: Tox21 or NCATS',
    'is_counter_screen': 'Boolean: True if assay is a counter-screen for false positives',
    'biological_process': 'Primary biological process measured: apoptosis, mitochondria, cytotoxicity',
    'endpoint_level': 'Endpoint classification: mechanistic (specific) or general_toxicity'
}

dict_df = pd.DataFrame([
    {'Column': k, 'Description': v} for k, v in data_dictionary.items()
])

dict_output_path = OUTPUT_DIR / 'data_dictionary.csv'
dict_df.to_csv(dict_output_path, index=False)
print(f"\nData dictionary saved to: {dict_output_path}")
print("\nColumn descriptions:")
for col, desc in data_dictionary.items():
    print(f"  {col}: {desc}")


Data dictionary saved to: output/data_dictionary.csv

Column descriptions:
  CID: PubChem Compound ID - unique identifier for each molecule
  SMILES: Canonical SMILES string representing molecular structure
  assay_name: Unique identifier for the bioassay (AID_xxx_description)
  assay_type: Type of assay: caspase, mitochondrial, or viability
  cell_line: Cell line used in the assay
  outcome: Activity classification: Active, Inactive, or Inconclusive
  score: Numeric activity score from PubChem (higher = more confident activity)
  source: Data source: Tox21 or NCATS
  is_counter_screen: Boolean: True if assay is a counter-screen for false positives
  biological_process: Primary biological process measured: apoptosis, mitochondria, cytotoxicity
  endpoint_level: Endpoint classification: mechanistic (specific) or general_toxicity


In [35]:
#  Export summary statistics
summary_stats = {
    'Total Records': len(merged_long),
    'Unique Compounds (CIDs)': merged_long['CID'].nunique(),
    'Unique SMILES': merged_long['SMILES'].nunique(),
    'Total Assays': merged_long['assay_name'].nunique(),
    'Primary Assays': merged_long[~merged_long['is_counter_screen']]['assay_name'].nunique(),
    'Counter-Screens': merged_long[merged_long['is_counter_screen']]['assay_name'].nunique(),
    'Active Outcomes': (merged_long['outcome'] == 'Active').sum(),
    'Inactive Outcomes': (merged_long['outcome'] == 'Inactive').sum(),
    'Inconclusive Outcomes': (merged_long['outcome'] == 'Inconclusive').sum(),
    'Active Rate (%)': round((merged_long['outcome'] == 'Active').mean() * 100, 2),
    'Missing SMILES': merged_long['SMILES'].isna().sum(),
    'Missing Scores': merged_long['score'].isna().sum(),
}

summary_df = pd.DataFrame([summary_stats]).T
summary_df.columns = ['Value']

summary_output_path = OUTPUT_DIR / 'summary_statistics.csv'
summary_df.to_csv(summary_output_path)
print(f"\nSummary statistics saved to: {summary_output_path}")
print("\n" + "="*70)
print("FINAL SUMMARY STATISTICS")
print("="*70)
print(summary_df)


Summary statistics saved to: output/summary_statistics.csv

FINAL SUMMARY STATISTICS
                            Value
Total Records            64516.00
Unique Compounds (CIDs)   8882.00
Unique SMILES             8880.00
Total Assays                22.00
Primary Assays              19.00
Counter-Screens              3.00
Active Outcomes           3755.00
Inactive Outcomes        53869.00
Inconclusive Outcomes     6892.00
Active Rate (%)              5.82
Missing SMILES               0.00
Missing Scores             279.00
